# ASMSA: Train AAE model with the tuned hyperparameters

**Previous steps**
- [02-prepare.ipynb](02-prepare.ipynb): Download and sanity check input files
- [03-tune.ipynb](03-tune.ipynb): Perform initial hyperparameter tuning for this molecule

**Next step**
- [05-plumed](05-plumed.ipynb): Generate Plumed input for the simulation
- [06-md.ipynb](06-md.ipynb): Use a trained model in MD simulation with Gromacs

In [ ]:
%cd ~/run
exec(open('inputs.py').read())

## Notebook setup

In [ ]:
threads = 2

import os
os.environ['OMP_NUM_THREADS']=str(threads)
import tensorflow as tf

# PyTorch favours OMP_NUM_THREADS in environment
import torch

# Tensorflow needs explicit cofig calls
tf.config.threading.set_inter_op_parallelism_threads(threads)
tf.config.threading.set_intra_op_parallelism_threads(threads)

In [ ]:
import tensorflow_probability as tfp
import matplotlib.pyplot as plt
import mdtraj as md
import numpy as np
import urllib.request
import keras_tuner
import asmsa.visualizer as visualizer
import asmsa

from tensorflow import keras
from asmsa.plot_training import LiveTrainingPlot
from asmsa.tuning_analyzer import TuningAnalyzer
from asmsa.helpers import create_gaussian_mixture_on_circle

## Load datasets
Load filtered trajectory datasets that were processed in **prepare.ipynb**. Trajectories are in internal coordinates format.

In [ ]:
# Get best HP from latest tuning 
# e.g: "analysis/xxx-yyy/"
# ... or don't specify, by default use the last analysis

analyzer = TuningAnalyzer()
analyzer.get_best_hp(num_trials=3)

In [ ]:
# Select HP to use by specifying trial_id
#  e.g: trial_id = '483883b929b3445bff6dee9759c4d50ee3a4ba7f0db22e665c49b5f942d9693b'
# ... or don't specify, by default use the trial with the lowest score
trial_id = ''

hps = None
for trial in analyzer.sorted_trials:
    if trial['trial_id'] == trial_id:
        hps = trial['hp']
    
if not hps:
    print(f'Could not find trial with specified ID, using one with the lowest score - {analyzer.sorted_trials[0]["trial_id"]}')
    hps = analyzer.sorted_trials[0]['hp']
    
print(hps)

In [ ]:
# load train dataset
X_train = tf.data.Dataset.load('01.datasets/intcoords/train')

# get batched version of dataset to feed to AAE model for training
X_train_batched = X_train.batch(hps['batch_size'],drop_remainder=True)

# get numpy version for visualization purposes
X_train_np = np.stack(list(X_train))
X_train_np.shape

In [ ]:
# load test dataset
X_test = tf.data.Dataset.load('01.datasets/intcoords/test')

# get batched version of dataset to feed to AAE model for prediction
X_test_batched = X_test.batch(hps['batch_size'],drop_remainder=True)

# get numpy version for testing purposes
X_test_np = np.stack(list(X_test))
X_test_np.shape

In [ ]:
X_val = tf.data.Dataset.load('01.datasets/intcoords/validate').batch(hps['batch_size'],drop_remainder=True)
X_val_np = np.stack(list(X_val))
X_val_np.shape

In [ ]:
# Pick best number of encoder and discriminator seeds from plots in 03-tune.ipynb 
best_enc_seed=32
best_disc_seed=32 

## Train

### Distribution prior
Train with common prior distributions. See https://www.tensorflow.org/probability/api_docs/python/tfp/distributions for all available distributions. 
These are selected with `{model_name}` parameter from [01-setup](01-setup.ipynb).

To be extra safe, the same distribution should have been used for hyperparameter tuning in [03-tune](03-tune.ipynb) but it usually works either.

In [ ]:
!ln -f ~/ASMSA/miscellaneus/mushroom_bw.png .

priors = {
    'norm': tfp.distributions.Normal(loc=0, scale=1),
    'unif': tfp.distributions.Uniform(),
    'mix': create_gaussian_mixture_on_circle(K=10, radius=1.5, scale_std=0.15),
    'mushroom': "mushroom_bw.png"
}

In [ ]:
model_name

In [ ]:
# Prepare model using the best hyperparameters from analysis

test = asmsa.AAEModel((X_train_np.shape[1],),
                       prior=priors[model_name],
                       hp=hps,
                       enc_seed=best_enc_seed,
                       disc_seed=best_disc_seed,
                       with_density=False
                      )
test.compile()

In [ ]:
test.summary()

In [ ]:
for layer in test.layers:
    print(f"\n== {layer.name} ==")
    if hasattr(layer, "layers"):  
        for sublayer in layer.layers:
            act = getattr(sublayer, "activation", None)
            act_name = act.__name__ if act is not None else "—"
            print(f"{sublayer.name:20s} | activation={act_name:10s} | params={sublayer.count_params()}")
    else:
        act = getattr(layer, "activation", None)
        act_name = act.__name__ if act is not None else "—"
        print(f"{layer.name:20s} | activation={act_name:10s} | params={layer.count_params()}")



In [ ]:
metric_groups = {
    "Autoencoder Loss": ["AE loss min", "val_AE loss min"],
    "Discriminator Loss": ["disc loss min", "val_disc loss min"],
}

# train it (can be repeated several times to add more epochs)
early_stop_cb = tf.keras.callbacks.EarlyStopping(
    monitor="val_AE loss min",
    min_delta=0.0001,
    patience=20,
    verbose=1,
    mode="min",
    restore_best_weights=True,
)

test.fit(X_train_batched, 
          epochs=1000,
          verbose=2, 
          validation_data=X_val,
          callbacks=[
              early_stop_cb,
              LiveTrainingPlot(metric_groups=metric_groups, freq=1),
              #visualizer.VisualizeCallback(test,freq=10,inputs=X_train_np[15000:25000],figsize=(12,3)) 
          ])

#Turn on the visualizer if you would like to see the latent space evolution every "freq" epochs. We advice to turn off the LiveTrainingPlot to avoid crowded output 

In [ ]:
# final visualization, pick a slice of the input data for demo purposes
#visualizer.Visualizer(figsize=(12,3)).make_visualization(testm.call_enc(X_train_np[15000:20000]).numpy())
# on test data
visualizer.Visualizer(figsize=(12,3)).make_visualization(test.call_enc(X_test_np).numpy())

In [ ]:
# load testing trajectory for further visualizations and computations
tr = md.load('./00.computations/x_test.xtc',top=conf)
idx=tr[0].top.select("name CA")

# for trivial cases like AlanineDipeptide
#idx=tr[0].top.select("element != H") 

tr.superpose(tr[0],atom_indices=idx)
geom = np.moveaxis(tr.xyz ,0,-1)
geom.shape

In [ ]:
# Rgyr and rmsd color coded in low dim (rough view), compute any other properties according to your needs

lows = test.call_enc(X_test_np).numpy()
rg = md.compute_rg(tr)
base = md.load(conf)
rmsd = md.rmsd(tr,base[0])
cmap = plt.get_cmap('rainbow')
plt.figure(figsize=(12,4))
plt.subplot(121)
plt.scatter(lows[:,0],lows[:,1],marker='.',c=rg,cmap=cmap,s=1)
plt.colorbar(cmap=cmap)
plt.title("Rg")
plt.subplot(122)
plt.scatter(lows[:,0],lows[:,1],marker='.',c=rmsd,cmap=cmap,s=1)
plt.colorbar(cmap=cmap)
plt.title("RMSD")
plt.show()

## Save the encoder and decoder models

In [ ]:
train_mean = np.loadtxt('01.datasets/intcoords/mean.txt',dtype=np.float32)
train_scale = np.loadtxt('01.datasets/intcoords/scale.txt',dtype=np.float32)

In [ ]:
import tf2onnx
import onnx2torch
import tempfile

def _convert_to_onnx(model, destination_path):
    input_tensor = model.layers[0]._input_tensor
    input_signature = tf.TensorSpec(
        name=input_tensor.name, shape=input_tensor.shape, dtype=input_tensor.dtype
    )
    output_name = model.layers[-1].name

    @tf.function(input_signature=[input_signature])
    def _wrapped_model(input_data):
        return {output_name: model(input_data)}

    tf2onnx.convert.from_function(
        _wrapped_model, input_signature=[input_signature], output_path=destination_path
    )

In [ ]:
model = test

In [ ]:
with tempfile.NamedTemporaryFile() as onnx:
    _convert_to_onnx(model.enc,onnx.name)
    torch_enc = onnx2torch.convert(onnx.name)

example_input = torch.randn([X_train_np.shape[1]])
traced_script_module = torch.jit.trace(torch_enc, example_input)

traced_script_module.save(f'encoder-{model_name}.pt')

In [ ]:
lenc = torch.jit.load(f'encoder-{model_name}.pt')
example_input = np.random.rand(10000,X_train_np.shape[1])
rtf = model.enc(example_input)
rpt = lenc(torch.tensor(example_input,dtype=torch.float32))

In [ ]:
maxerr = np.max(np.abs(rtf - rpt.detach().numpy()))
maxerr

In [ ]:
with tempfile.NamedTemporaryFile() as onnx:
    _convert_to_onnx(model.dec,onnx.name)
    torch_dec = onnx2torch.convert(onnx.name)

example_input = torch.randn([2])
traced_script_module = torch.jit.trace(torch_dec, example_input)

traced_script_module.save(f'decoder-{model_name}.pt')

In [ ]:
ldec = torch.jit.load(f'decoder-{model_name}.pt')
example_input = np.random.rand(10000,2)
rtf = model.dec(example_input)
rpt = ldec(torch.tensor(example_input,dtype=torch.float32))

In [ ]:
err = np.abs(rtf - rpt.detach().numpy())
rerr = err/np.abs(train_mean)
np.max(err),np.max(rerr)

In [ ]:
%mkdir -p 03.model
!mv encoder-{model_name}.pt decoder-{model_name}.pt 03.model